In [1]:
import pandas as pd

In [11]:
df = pd.read_clipboard()

In [12]:
#make a date column from Year and Month
df['Date'] = pd.to_datetime(df[['Year', 'Month']].assign(Day=1))
#find duplicated months or missing months
df['Month_Year'] = df['Date'].dt.to_period('M')
duplicates = df[df.duplicated('Month_Year', keep=False)]
missing_months = pd.date_range(start=df['Date'].min(), end=df['Date'].max(), freq='M').to_period('M')
missing_months = missing_months[~missing_months.isin(df['Month_Year'])]
print("Duplicates:")
print(duplicates[['Month_Year', 'Date']])
print("Missing Months:")
print(missing_months)


Duplicates:
   Month_Year       Date
28    2025-01 2025-01-01
29    2025-02 2025-02-01
30    2025-03 2025-03-01
31    2025-01 2025-01-01
32    2025-02 2025-02-01
33    2025-03 2025-03-01
Missing Months:
PeriodIndex(['2023-07', '2023-09', '2024-04'], dtype='period[M]')


In [8]:
duplicates, 

(Empty DataFrame
 Columns: [Name, Year, Month, Value, Date, Month_Year]
 Index: [],)

In [7]:
missing_months

DatetimeIndex(['2023-06-30', '2023-07-31', '2023-08-31', '2023-09-30',
               '2023-10-31', '2023-11-30', '2023-12-31', '2024-01-31',
               '2024-02-29', '2024-03-31',
               ...
               '2040-02-29', '2040-03-31', '2040-04-30', '2040-05-31',
               '2040-06-30', '2040-07-31', '2040-08-31', '2040-09-30',
               '2040-10-31', '2040-11-30'],
              dtype='datetime64[ns]', length=210, freq=None)

In [13]:
"""
Debug optimization data structure and display_performance_metrics function
"""
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))

import pandas as pd
import numpy as np
from app.course import Course
from app.optimization import CourseOptimizer, create_course_segments_from_course

def test_optimization_data():
    """Test optimization data structure to fix display issues."""
    
    print("🔍 Testing Optimization Data Structure")
    print("=" * 50)
    
    # Create mock course data
    mock_data = pd.DataFrame({
        'lat': np.linspace(40.0, 40.1, 100),
        'lon': np.linspace(-74.0, -73.9, 100), 
        'altitude': 100 + np.sin(np.linspace(0, 4*np.pi, 100)) * 50,
        'cumulative_distance': np.linspace(0, 10000, 100),
        'distance': np.full(100, 100),
        'gradient': np.sin(np.linspace(0, 4*np.pi, 100)) * 0.05,
        'bearing': np.linspace(0, 360, 100),
        'height_gain': np.random.uniform(0, 5, 100),
        'height_loss': np.random.uniform(0, 5, 100)
    })
    
    course = Course(mock_data)
    
    # Segment with same params as 01 page
    course.auto_segmentation(
        max_clusters=5,
        gradient_weight=0.5,
        bearing_weight=0.5, 
        smoothing_window=20
    )
    
    print(f"✅ Course segmented into {len(course.course_segments)} segments")
    
    # Create course segments and run optimization
    course_segments = create_course_segments_from_course(course)
    optimizer = CourseOptimizer(critical_power=300, w_prime=25000)
    
    # Test constant power optimization
    optimal_power, total_time, detailed_results = optimizer.optimize_constant_power(course_segments)
    
    # Create the simulation results dict that _analyze_detailed_simulation expects
    simulation_results = {
        'total_time': total_time,
        'optimal_power': optimal_power,
        'w_remaining': detailed_results['w_remaining'].iloc[-1] if 'w_remaining' in detailed_results.columns else 25000
    }
    
    granular_analysis = optimizer._analyze_detailed_simulation(detailed_results, simulation_results, course_segments)
    
    print(f"\n📊 Optimization Results:")
    print(f"   Optimal Power: {optimal_power:.0f}W")
    print(f"   Total Time: {total_time/60:.1f} min")
    print(f"   Detailed Results Shape: {detailed_results.shape}")
    
    print(f"\n🔍 Granular Analysis Keys:")
    for key, value in granular_analysis.items():
        if isinstance(value, dict):
            print(f"   📁 {key}: {list(value.keys())}")
        else:
            print(f"   📊 {key}: {value}")
    
    # Check segment distances
    print(f"\n📏 Segment Distance Analysis:")
    distances = course.course_segments['distance_sum']
    print(f"   Distances: {distances.tolist()}")
    print(f"   Zero distances: {(distances == 0).sum()}")
    print(f"   Total: {distances.sum():.2f} km")
    
    return granular_analysis, course.course_segments

# Run the test
analysis, segments = test_optimization_data()

🔍 Testing Optimization Data Structure
running process segment


c:\Users\chrisuberti\Anaconda3\lib\site-packages\sklearn\cluster\_agglomerative.py:542: ClusterWarning: scipy.cluster: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  out = hierarchy.linkage(X, method=linkage, metric=affinity)


✅ Course segmented into 5 segments
Using traditional aggregated segmentation format with 5 segments
Optimization converged to 315W (target W' utilization: 85%)
Achieved W' utilization: 85.0% (target: 85%)
Optimization converged to 315W (target W' utilization: 85%)
Achieved W' utilization: 85.0% (target: 85%)


TypeError: _analyze_detailed_simulation() missing 1 required positional argument: 'course_segments'

In [ ]:
# Test for zero distance segments issue
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))

import pandas as pd
import numpy as np
from app.course import Course

# Create test course with more realistic data
mock_data = pd.DataFrame({
    'lat': np.linspace(40.0, 40.1, 200),  # More data points
    'lon': np.linspace(-74.0, -73.9, 200),
    'altitude': 100 + np.sin(np.linspace(0, 4*np.pi, 200)) * 50,
    'cumulative_distance': np.linspace(0, 20000, 200),  # 20km total
    'distance': np.full(200, 100),  # 100m per point
    'gradient': np.sin(np.linspace(0, 4*np.pi, 200)) * 0.05,
    'bearing': np.linspace(0, 360, 200),
    'height_gain': np.random.uniform(0, 5, 200),
    'height_loss': np.random.uniform(0, 5, 200)
})

course = Course(mock_data)

# Test segmentation with default parameters from 01 page
print("🔧 Testing segmentation with 01 page defaults...")
course.auto_segmentation(
    max_clusters=5,
    gradient_weight=0.5,
    bearing_weight=0.5,
    smoothing_window=20
)

print(f"✅ Segmented into {len(course.course_segments)} segments")
print("\n📊 Segment Details:")
print(course.course_segments[['segment_id', 'distance_sum', 'gradient_mean', 'segment_sequence']].round(3))